In [1]:
!pip3 -q install --upgrade google-cloud-documentai google-cloud-storage google-cloud-documentai-toolbox


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-api-python-client 1.8.0 requires google-api-core<2dev,>=1.13.0, but you have google-api-core 2.21.0 which is incompatible.


# OCR Parser

In [6]:
from google.api_core.client_options import ClientOptions
from google.cloud import documentai


PROJECT_ID = ""
LOCATION = "us"  # Format is 'us' or 'eu'
PROCESSOR_ID = ""

# The local file in your current working directory
FILE_PATH = "data/naf23mini.pdf"
# Refer to https://cloud.google.com/document-ai/docs/file-types
# for supported file types
MIME_TYPE = "application/pdf"

In [7]:
# Instantiates a client
docai_client = documentai.DocumentProcessorServiceClient(
    client_options=ClientOptions(api_endpoint=f"{LOCATION}-documentai.googleapis.com")
)

In [8]:
# The full resource name of the processor, e.g.:
# projects/project-id/locations/location/processor/processor-id
# You must create new processors in the Cloud Console first
RESOURCE_NAME = docai_client.processor_path(PROJECT_ID, LOCATION, PROCESSOR_ID)

In [11]:
# Read the file into memory
with open(FILE_PATH, "rb") as image:
    image_content = image.read()

# Load Binary Data into Document AI RawDocument Object
raw_document = documentai.RawDocument(content=image_content, mime_type=MIME_TYPE)

# Configure the process request
request = documentai.ProcessRequest(name=RESOURCE_NAME, raw_document=raw_document)

# Use the Document AI client to process the sample form
result = docai_client.process_document(request=request)

document_object = result.document
print("Document processing complete.")
print(f"Text: {document_object.text}")

In [10]:
type(result)

google.cloud.documentai_v1.types.document_processor_service.ProcessResponse

In [12]:
type(document_object)

google.cloud.documentai_v1.types.document.Document

In [17]:
# document_object

In [26]:
print(document_object.text)

National Ataxia Foundation
Table of Contents
December 31, 2023 and 2022
Page No.
3
Independent Auditor's Report
Financial Statements
Statements of Financial Position
Statements of Activities
Statements of Functional Expenses
Statements of Cash Flows
Notes to the Financial Statements
2
160
7
9
11
12
Assets
Current Assets
Cash and cash equivalents
Accounts receivable
Prepaid expenses
Total Current Assets
Noncurrent Assets
Right-of-use asset
Investments
Total Noncurrent Assets
Total Assets
Liabilities and Net Assets
Current Liabilities
Accounts payable
National Ataxia Foundation
Statements of Financial Position
December 31, 2023 and 2022
2023
2022
$ 1,008,716
6,438
255,729
$ 1,969,164
55,331
187,070
1,270,883
2,211,565
57,320
89,034
2,788,623
2,707,996
2,845,943
2,797,030
$ 4,116,826 $ 5,008,595
$
298,827 $ 440,809
Accrued payroll and related expenses
35,140
29,226
Deferred revenue
109,810
22,291
Operating lease liability, current portion
32,750
31,548
Total Current Liabilities
476,527
52

In [28]:
document_object.chunked_document

In [30]:
for page in document_object.pages:
    print(page.page_number)
    print(f"\nFound {len(page.tables)} table(s):")

1

Found 0 table(s):
2

Found 0 table(s):
3

Found 0 table(s):
4

Found 0 table(s):


# Form Parser

In [31]:
from google.api_core.client_options import ClientOptions
from google.cloud import documentai
from os.path import splitext
from typing import List, Sequence
from google.cloud import documentai
import pandas as pd


PROJECT_ID = 
LOCATION = "us"  # Format is 'us' or 'eu'
PROCESSOR_ID = 

# The local file in your current working directory
FILE_PATH = "data/naf23mini.pdf"
# Refer to https://cloud.google.com/document-ai/docs/file-types
# for supported file types
MIME_TYPE = "application/pdf"


In [34]:
def online_process(
    project_id: str,
    location: str,
    processor_id: str,
    file_path: str,
    mime_type: str,
) -> documentai.Document:
    """
    Processes a document using the Document AI Online Processing API.
    """

    opts = {"api_endpoint": f"{location}-documentai.googleapis.com"}

    # Instantiates a client
    documentai_client = documentai.DocumentProcessorServiceClient(client_options=opts)

    # The full resource name of the processor, e.g.:
    # projects/project-id/locations/location/processor/processor-id
    # You must create new processors in the Cloud Console first
    resource_name = documentai_client.processor_path(project_id, location, processor_id)

    # Read the file into memory
    with open(file_path, "rb") as image:
        image_content = image.read()

        # Load Binary Data into Document AI RawDocument Object
        raw_document = documentai.RawDocument(
            content=image_content, mime_type=mime_type
        )

        # Configure the process request
        request = documentai.ProcessRequest(
            name=resource_name, raw_document=raw_document
        )

        # Use the Document AI client to process the sample form
        result = documentai_client.process_document(request=request)

        return result.document
    
document = online_process(
    project_id=PROJECT_ID,
    location=LOCATION,
    processor_id=PROCESSOR_ID,
    file_path=FILE_PATH,
    mime_type=MIME_TYPE,
)


In [52]:
def get_table_data(
    rows: Sequence[documentai.Document.Page.Table.TableRow], text: str
) -> List[List[str]]:
    """
    Get Text data from table rows
    """
    all_values: List[List[str]] = []
    for row in rows:
        current_row_values: List[str] = []
        for cell in row.cells:
            current_row_values.append(
                text_anchor_to_text(cell.layout.text_anchor, text)
            )
        all_values.append(current_row_values)
    return all_values

def text_anchor_to_text(text_anchor: documentai.Document.TextAnchor, text: str) -> str:
    """
    Document AI identifies table data by their offsets in the entirity of the
    document's text. This function converts offsets to a string.
    """
    response = ""
    # If a text segment spans several lines, it will
    # be stored in different text segments.
    for segment in text_anchor.text_segments:
        start_index = int(segment.start_index)
        end_index = int(segment.end_index)
        response += text[start_index:end_index]
    return response.strip().replace("\n", " ")

In [55]:
for page in document.pages:
    print(len(page.tables), 'tables found' )
    
    # if len(page.tables)==0:
        # print(page.tokens)
    # print(page.page_number)
    for index, table in enumerate(page.tables):
        header_row_values = get_table_data(table.header_rows, document.text)
        body_row_values = get_table_data(table.body_rows, document.text)
        
        df = pd.DataFrame(
            data=body_row_values,
            columns=pd.MultiIndex.from_arrays(header_row_values),
        )
        print(f"Page {page.page_number} - Table {index}")
        display(df)
        print(header_row_values)

0 tables found
5 tables found
Page 2 - Table 0


,Total Liabilities,"501,880 581,978"
0,Net Assets Without donor restriction,
1,Board designated - operating reserve,"396,499 420,985"
2,Undesignated,"537,880 1,963,826"
3,Total Net Assets Without Donor Restriction,"934,379 2,384,811"
4,With donor restriction,"2,680,567 2,041,806"
5,Total Net Assets,"3,614,946 4,426,617"
6,Total Liabilities and Net Assets,"$ 4,116,826 $ 5,008,595"


[['Total Liabilities', '501,880 581,978']]
Page 2 - Table 1


,Cash and cash equivalents,"$ 1,008,716","$ 1,969,164"
0,Accounts receivable,"6,438","55,331"
1,Prepaid expenses,"255,729","187,070"
2,Total Current Assets,"1,270,883","2,211,565"


[['Cash and cash equivalents', '$ 1,008,716', '$ 1,969,164']]
Page 2 - Table 2


...

[['3,592 1,332 Bank and credit card fees', '', '4,924', '43,689', '48,613']]
Page 3 - Table 1


,Research,Education and Service,Drug Development Collaborative,Total Program Services,Management and General,Fundraising,Total
0,"$ 494,104","$ 362,830","$ 162,033","$ 1,018,967","$ 138,082","$ 293,427","$ 1,450,476"
1,"38,208","27,159","12,319","77,686","9,732","21,710","109,128"
2,"45,455","52,132","13,763","111,350","21,428","39,024","171,802"
3,"577,767","442,121","188,115","1,208,003","169,242","354,161","1,731,406"
4,"1,332","3,592",,"4,924","43,689",,"48,613"


[['Research', 'Education and Service', 'Drug Development Collaborative', 'Total Program Services', 'Management and General', 'Fundraising', 'Total']]
Page 3 - Table 2


,Salaries and wages,"$ 494,104","$ 362,830","$ 162,033",$
0,Payroll taxes,"38,208","27,159","12,319",
1,Fringe benefits,"45,455","52,132","13,763",
2,Total Salaries and Related Expenses,"577,767","442,121","188,115",


[['Salaries and wages', '$ 494,104', '$ 362,830', '$ 162,033', '$']]
0 tables found
